# Poetry Lab — первый реальный train/sample

Этот notebook запускает **масочную дискретную диффузию** на русской предобученной MLM. Источник bootstrap-данных — стихотворные отрывки Пушкина с меткой `PD-old-70` в Викитеке; это **финальные тексты, не авторские черновики**. Результат после короткого обучения — технический прототип, а не доказательство художественного качества.

Перед запуском выберите **Runtime → Change runtime type → GPU**. Подключение Google Drive нужно лишь для сохранения весов после выключения Colab. Код и исходные ссылки остаются в Drive пользователя; корпус и веса не пушатся в публичный GitHub.


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Включите GPU в настройках runtime и повторите ячейку')
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi


In [ ]:
!git clone https://github.com/aisarus/Poetry.git /content/Poetry
%cd /content/Poetry
!python -m pip -q install transformers
!python -m unittest discover -q


## Сохранение чекпойнта

Google Drive попросит предоставить доступ в интерфейсе Colab. Если не хотите подключать Drive, замените `ROOT` на `/content/PoetryLab`: тогда файлы исчезнут при завершении runtime, их потребуется скачать вручную.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/PoetryLab')
(ROOT / 'data').mkdir(parents=True, exist_ok=True)
(ROOT / 'checkpoints').mkdir(parents=True, exist_ok=True)
(ROOT / 'samples').mkdir(parents=True, exist_ok=True)


In [ ]:
import json, subprocess, sys
corpus = ROOT / 'data' / 'pushkin_pd.jsonl'
if not corpus.exists():
    subprocess.run([sys.executable, 'wikisource_corpus.py', '--max-pages', '150',
                    '--out', str(corpus)], check=True)
rows = [json.loads(line) for line in corpus.read_text(encoding='utf-8').splitlines() if line.strip()]
print('Отрывков:', len(rows), 'страниц:', len({r['source_url'] for r in rows}))
if len(rows) < 30:
    raise RuntimeError('Корпус слишком мал: проверьте ответы Викитеки и парсер')
print('Первый источник:', rows[0]['source_url'])


## Bootstrap-обучение

Первый прогон ограничен 250 отрывками и одной эпохой, чтобы проверить пайплайн и получить чекпойнт. Для хороших стихов этого **недостаточно**. После проверки увеличивайте объём/эпохи, собирайте принятые человеком пары `draft → revision` и дообучайте через `--data` с `--base-model` равным этому чекпойнту. Отложенные страницы и темы нельзя использовать в обучении при честной оценке.


In [ ]:
checkpoint = ROOT / 'checkpoints' / 'bootstrap-v1'
subprocess.run([sys.executable, 'masked_diffusion.py', 'train', '--corpus', str(corpus),
                '--out', str(checkpoint), '--epochs', '1', '--limit', '250',
                '--max-length', '256', '--accumulation', '8'], check=True)
print('Чекпойнт:', checkpoint)
print(json.loads((checkpoint / 'diffusion_config.json').read_text(encoding='utf-8')))


In [ ]:
sample = ROOT / 'samples' / 'first.txt'
subprocess.run([sys.executable, 'masked_diffusion.py', 'sample',
                '--checkpoint', str(checkpoint), '--topic', 'Тёплая ирония случайной встречи',
                '--stanzas', '2', '--steps', '16', '--out', str(sample)], check=True)
print('Сохранено:', sample)


Следующая проверка — **ручная**: прочитать результат вслух, отметить ритм, рифму, смысл и слабые образы. Нельзя выдавать loss или список фонетических подсказок за оценку художественности. Для слепого A/B с Ollama baseline используется `blind_eval.py` в репозитории.
